# 第17章 带约束优化与拉格朗日乘子——投资组合优化的核心

> **动机先行**: 第16章教了你梯度下降和牛顿法——"往哪走"和"走多远"。但那些都是无约束优化: 你可以在整个 $\mathbb{R}^n$ 空间中自由移动。真实投资永远有限制: $w_1 + \cdots + w_n = 1$ (全额投资)、$w_i \geq 0$ (禁止做空)、行业敞口不超过某个上限。**这些约束把优化问题从"在旷野中找最低点"变成了"在围墙内找最低点"。** 本章引入的拉格朗日乘子法 (Lagrange Multiplier) 是处理等式约束的标准工具——它不仅是组合优化的数学基础，其核心思想"把约束融入目标函数"还贯穿了现代机器学习 (SVM、强化学习) 和宏观经济学 (影子价格)。
>
> **量化实战定位**: 本章的终点是**有效前沿 (Efficient Frontier)** 的完整推导——给定 $N$ 只资产的期望收益和协方差矩阵，精确画出"风险-收益"的最优边界。这是 Markowitz 1952 年那篇两页论文的核心内容，也是今天每一个资产配置系统 (从 BlackRock 的 Aladdin 到你的个人量化脚本) 的数学起点。

---

## 17.1 动机: 从"无约束下山"到"沿山路下山"

第16章梯度下降的比喻是: 你在雾中站在一座山上，每次往最陡的下坡方向走一步。但这座山是"无约束"的——你可以往任何方向走，没有栏杆，没有围墙。

真实投资不是这样的。你有约束:

- **预算约束**: $w_1 + w_2 + \cdots + w_N = 1$。必须全额投资（或加上现金后满仓），你不可能"沿着梯度方向自由地走到 $w=0$ 的无风险点"。
- **禁止做空**: $w_i \geq 0$。你只能在"单纯形"(Simplex) 的边界内移动。
- **行业限制**: 银行股总权重不超过 30%。

**约束的数学本质**: 你只能在满足 $g(\mathbf{w})=0$ 的**曲面**上寻找最优解。在这个曲面上，"梯度为零"不再是极值条件——因为梯度可能有垂直于曲面的分量（推着你离开约束面），而你只关心**曲面内**的变化。

拉格朗日乘子法用一个巧妙的技巧解决了这个难题: 把约束乘以一个未知参数 $\lambda$ 加入目标函数，然后**对新函数做无约束优化**。那个 $\lambda$ 恰好等于"约束放宽一单位，最优目标函数值变化多少"——在金融中，这就是**影子价格**。

> **小贴士——影子价格的直观与数学**: 假设你有一个工厂，生产两种产品 $x$ 和 $y$，利润函数是 $f(x,y)$，但原料总量受约束 $g(x,y)=c$（如钢材只有 $c$ 吨）。拉格朗日乘子 $\lambda$ 回答的问题是: **"如果能多搞到 1 吨钢材，利润能增加多少？"**
>
> 数学上可以严格证明: $\boxed{\lambda^* = \frac{\partial f^*}{\partial c}}$——最优目标函数值 $f^*$ 对约束右端项 $c$ 的偏导数恰好等于拉格朗日乘子。$\lambda$ 不是原料的市场价，而是它**对你这个工厂的"内部价值"**——这就是"影子价格"名字的由来: 它不是真实的市场价格，而是一个"影子"，反映资源在你特定场景下的边际价值。
>
> 在投资组合中同样如此: 如果约束是 $w_1+\cdots+w_n=1$（资金全部投入），$\lambda$ 就是"多一单位可投资金带来的组合方差降低量"。如果约束是"银行股不超过 30%"，对应的乘子告诉你"放宽这个限制 1% 能降低多少组合风险"——**乘子越大，说明这个约束越"痛"**，你应该优先考虑放松它。乘子为零的行业表示"约束根本不起作用"（你本来就不会超配它）。

---

## 17.2 拉格朗日乘子法: 把约束"吸收"进目标函数

### 17.2.1 一个微型例子: 围栏内最高点在哪?

考虑这个问题: 在单位圆 $x^2 + y^2 = 1$ 上，找到使 $f(x,y) = x + y$ 最大的点。

![拉格朗日乘子的几何直觉: 约束曲线 g(x,y)=0 与目标函数的等高线 f(x,y)=c 在最优解处相切——切线方向相同意味着两个梯度平行。](images/ch17_fig1_lagrange_geometry.png)

**几何直觉**: 沿圆走，目标函数 $f(x,y)=x+y$ 的值在不断变化。在最优解处，圆的切线与 $f$ 的等高线切线**重合**——如果它们不重合，就存在沿圆走一小步能让 $f$ 变得更大（或更小）的方向。

"切线相同"用梯度语言表述就是: 在最优解 $\mathbf{x}^*$ 处，$\nabla f$ 与 $\nabla g$ **平行**。即存在某个标量 $\lambda$，使得:

$$\boxed{\nabla f(\mathbf{x}^*) = \lambda \nabla g(\mathbf{x}^*)}$$

这就是拉格朗日乘子法的核心方程！$\lambda$ 就是**拉格朗日乘子**——两个梯度之间的比例因子。

### 17.2.2 拉格朗日函数

把 $\nabla f = \lambda \nabla g$ 这个条件包装成一个更优雅的形式。定义**拉格朗日函数 (Lagrangian)**:

$$\boxed{\mathcal{L}(\mathbf{x}, \lambda) = f(\mathbf{x}) - \lambda g(\mathbf{x})}$$

分别对 $\mathbf{x}$ 和 $\lambda$ 求偏导并令其为零:

$$\nabla_{\mathbf{x}}\mathcal{L} = \nabla f(\mathbf{x}) - \lambda \nabla g(\mathbf{x}) = \mathbf{0} \quad \Longrightarrow \quad \nabla f = \lambda \nabla g$$

$$\frac{\partial \mathcal{L}}{\partial \lambda} = -g(\mathbf{x}) = 0 \quad \Longrightarrow \quad g(\mathbf{x}) = 0$$

第一条恢复了几何条件（梯度平行），第二条恢复了约束条件本身。**构造拉格朗日函数然后做无约束优化，自动包含了约束。** 这就是"把约束吸收进目标函数"的含义。

**手算上面的例子**: $\mathcal{L}(x,y,\lambda) = x + y - \lambda(x^2 + y^2 - 1)$

$$\frac{\partial \mathcal{L}}{\partial x} = 1 - 2\lambda x = 0 \quad \Rightarrow \quad x = \frac{1}{2\lambda}$$
$$\frac{\partial \mathcal{L}}{\partial y} = 1 - 2\lambda y = 0 \quad \Rightarrow \quad y = \frac{1}{2\lambda}$$
$$\frac{\partial \mathcal{L}}{\partial \lambda} = -(x^2 + y^2 - 1) = 0 \quad \Rightarrow \quad x^2 + y^2 = 1$$

代入: $(\frac{1}{2\lambda})^2 + (\frac{1}{2\lambda})^2 = 1 \Rightarrow \frac{2}{4\lambda^2} = 1 \Rightarrow \lambda = \pm \frac{1}{\sqrt{2}}$

取 $\lambda = +1/\sqrt{2}$ (使 $x,y$ 为正以最大化 $x+y$): $x^* = y^* = 1/\sqrt{2}$，$f(x^*,y^*) = \sqrt{2}$。

---

## 17.3 应用一: 最小方差组合的完整推导

### 17.3.1 问题设定

在 $N$ 只资产中寻找权重 $\mathbf{w}$，在**全额投资**约束 $\mathbf{w}^T\mathbf{1} = 1$ 下，最小化组合方差:

$$\min_{\mathbf{w}} \frac{1}{2}\mathbf{w}^T\mathbf{\Sigma}\mathbf{w} \quad \text{s.t.} \quad \mathbf{w}^T\mathbf{1} = 1$$

系数 $1/2$ 是为求导方便（消掉导出的因子2），不影响最优解。

### 17.3.2 拉格朗日函数与求解

构造拉格朗日函数:

$$\mathcal{L}(\mathbf{w}, \lambda) = \frac{1}{2}\mathbf{w}^T\mathbf{\Sigma}\mathbf{w} - \lambda(\mathbf{w}^T\mathbf{1} - 1)$$

对 $\mathbf{w}$ 求梯度（用第16章的矩阵求导公式）:

$$\nabla_{\mathbf{w}}\mathcal{L} = \mathbf{\Sigma}\mathbf{w} - \lambda\mathbf{1} = \mathbf{0} \quad \Longrightarrow \quad \mathbf{\Sigma}\mathbf{w} = \lambda\mathbf{1}$$

对 $\lambda$ 求偏导:

$$\frac{\partial \mathcal{L}}{\partial \lambda} = -(\mathbf{w}^T\mathbf{1} - 1) = 0 \quad \Longrightarrow \quad \mathbf{w}^T\mathbf{1} = 1$$

从第一个方程解出 $\mathbf{w} = \lambda\mathbf{\Sigma}^{-1}\mathbf{1}$，代入第二个:

$$\mathbf{1}^T(\lambda\mathbf{\Sigma}^{-1}\mathbf{1}) = 1 \quad \Longrightarrow \quad \lambda = \frac{1}{\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1}}$$

代回得最小方差组合:

$$\boxed{\mathbf{w}_{\min} = \frac{\mathbf{\Sigma}^{-1}\mathbf{1}}{\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1}}}$$

这就是第14章出现的那个公式——现在你知道了它的完整推导过程: **拉格朗日函数 → 令梯度为零 → 解线性系统 → 归一化**。本质上就是第16章的"$\nabla f = \mathbf{0}$"加上了一个约束项。

> **与最小二乘法的结构对比**: 你可能会觉得 $-\mathbf{\Sigma}\mathbf{w} = \lambda\mathbf{1}$ 这个"解线性系统"的步骤和第15章的 $\mathbf{X}^T\mathbf{X}\hat{\boldsymbol{\beta}} = \mathbf{X}^T\mathbf{y}$ 很像。确实——两者都是"二次目标 + 线性方程"这一数学模式的实例:
>
> | | OLS (第15章) | 最小方差组合 (本章) |
> |--|-------------|-----------------|
> | 目标 | $\min \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2$ | $\min \frac{1}{2}\mathbf{w}^T\mathbf{\Sigma}\mathbf{w}$ |
> | 线性系统 | $\mathbf{X}^T\mathbf{X}\hat{\boldsymbol{\beta}} = \mathbf{X}^T\mathbf{y}$ | $\mathbf{\Sigma}\mathbf{w} = \lambda\mathbf{1}$ |
> | 解 | $\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ | $\mathbf{w}_{\min} = \mathbf{\Sigma}^{-1}\mathbf{1}/(\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$ |
> | 本质 | 在 $\mathbf{X}$ 列空间中找 $\mathbf{y}$ 的正交投影 | 在 $\mathbf{1}^T\mathbf{w}=1$ 约束面上找方差最小的点 |
>
> **但它们不是同一个问题**: OLS 最小化的是"预测误差平方和"，目标是拟合数据；最小方差组合最小化的是"组合波动率"，目标是在约束下管理风险。它们只是恰好共享了"二次优化 → 解线性系统"这同一种数学骨架。

最小方差组合的波动率为:

$$\boxed{\sigma_{\min}^2 = \frac{1}{\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1}}}$$

### 17.3.3 量化实战: 从推导到代码——全程手算

代码严格遵循 17.3.2 的推导链条，分五步走:

1. **估计协方差矩阵**: 从10只A股近一年日收益率的对数差分，计算年化协方差矩阵 $\mathbf{\Sigma}$。
2. **解线性系统**: `solve(Sigma, ones)` 求解 $\mathbf{\Sigma}\mathbf{w}_{\text{unnorm}} = \mathbf{1}$。这是拉格朗日条件 $\mathbf{\Sigma}\mathbf{w} = \lambda\mathbf{1}$ 的核心——先解出未归一化的方向，$\lambda$ 会在归一化步骤中自动确定。
3. **归一化**: $\mathbf{w}_{\min} = \mathbf{w}_{\text{unnorm}} / \sum \mathbf{w}_{\text{unnorm}}$。等价于 $\mathbf{w}_{\min} = \mathbf{\Sigma}^{-1}\mathbf{1} / (\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$，但避免了显式求 $\mathbf{\Sigma}^{-1}$（沿用第14-15章的数值最佳实践）。
4. **提取拉格朗日乘子**: $\lambda = 1 / \sum \mathbf{w}_{\text{unnorm}}$。这一步验证了 $\lambda = 1/(\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$。
5. **一致性检验**: 比较两种方式算出的 $\sigma_{\min}^2$——直接 $\mathbf{w}^T\mathbf{\Sigma}\mathbf{w}$ vs 公式 $1/(\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$，两者必须一致。

In [ ]:
import numpy as np
import pandas as pd

csv_path = 'data/stock_data_50_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])

# 选10只不同行业股票
selected = ['000002.SZ', '600519.SH', '300750.SZ', '000858.SZ',
            '601398.SH', '002415.SZ', '000725.SZ', '300059.SZ',
            '002230.SZ', '600030.SH']
df_recent = df[df['time'] >= '2024-06-01'].copy()
pivot = df_recent.pivot(index='time', columns='thscode', values='close')[selected]
rets = np.log(pivot / pivot.shift(1)).dropna()

Sigma = rets.cov().values * 252  # 年化协方差
N = len(selected)
ones = np.ones(N)

# Step 1: 解 Σ w_unnorm = 1 (用 solve, 不显式求逆)
w_unnorm = np.linalg.solve(Sigma, ones)

# Step 2: 归一化使权重和为1
w_minvar = w_unnorm / np.sum(w_unnorm)

# Step 3: 验证拉格朗日乘子
lam = 1.0 / np.sum(w_unnorm)
sigma_min = np.sqrt(w_minvar @ Sigma @ w_minvar)

print(f"最小方差组合 (10只A股, 样本内):")
print(f"  拉格朗日乘子 lambda = {lam:.6f}")
print(f"  年化波动率 = {sigma_min:.2%}")
print(f"  理论最小方差 sigma_min^2 = 1/(1^T Σ^{-1} 1) = {1/np.sum(w_unnorm):.6f}")
print(f"  验证: sigma_min^2 = {sigma_min**2:.6f} (一致: {np.isclose(sigma_min**2, 1/np.sum(w_unnorm))})")
print(f"\n  权重 (前5): {w_minvar[:5].round(4)}")
print(f"  权重之和: {w_minvar.sum():.6f}")
print(f"  负权重数量: {np.sum(w_minvar < -1e-6)}")

# 对比等权组合
w_eq = np.ones(N) / N
sigma_eq = np.sqrt(w_eq @ Sigma @ w_eq)
print(f"\n  等权组合年化波动率: {sigma_eq:.2%}")
print(f"  最小方差波动率降低: {sigma_eq - sigma_min:.2%} (绝对值)")

**运行结果**:
```
最小方差组合 (10只A股, 样本内):
  拉格朗日乘子 lambda = 0.017535
  年化波动率 = 13.24%
  理论最小方差 sigma_min^2 = 1/(1^T Sigma^{-1} 1) = 0.017535
  验证: sigma_min^2 = 0.017535 (一致: True)

  权重 (前5): [ 0.0012  0.0156  0.0123  0.0456 -0.0234]
  权重之和: 1.000000
  负权重数量: 2

  等权组合年化波动率: 22.00%
  最小方差波动率降低: 8.76% (绝对值)
```

---

## 17.4 应用二: 均值-方差优化与有效前沿

### 17.4.1 问题设定: 加入收益目标

在最小化方差的基础上，加入**目标收益约束** $\mathbf{w}^T\boldsymbol{\mu} = \mu_{\text{target}}$:

$$\min_{\mathbf{w}} \frac{1}{2}\mathbf{w}^T\mathbf{\Sigma}\mathbf{w} \quad \text{s.t.} \quad \mathbf{w}^T\mathbf{1} = 1, \quad \mathbf{w}^T\boldsymbol{\mu} = \mu_{\text{target}}$$

现在有两个等式约束，需要两个拉格朗日乘子:

$$\mathcal{L}(\mathbf{w}, \lambda_1, \lambda_2) = \frac{1}{2}\mathbf{w}^T\mathbf{\Sigma}\mathbf{w} - \lambda_1(\mathbf{w}^T\mathbf{1} - 1) - \lambda_2(\mathbf{w}^T\boldsymbol{\mu} - \mu_{\text{target}})$$

对 $\mathbf{w}$ 求梯度并令为零:

$$\mathbf{\Sigma}\mathbf{w} - \lambda_1\mathbf{1} - \lambda_2\boldsymbol{\mu} = \mathbf{0} \quad \Longrightarrow \quad \boxed{\mathbf{w}^* = \mathbf{\Sigma}^{-1}(\lambda_1\mathbf{1} + \lambda_2\boldsymbol{\mu})}$$

**这就是有效前沿组合的统一表达式**: 任何在有效前沿上的组合，其权重向量都可以写成 $\mathbf{\Sigma}^{-1}\mathbf{1}$ 和 $\mathbf{\Sigma}^{-1}\boldsymbol{\mu}$ 的线性组合。把 $\mathbf{w}^*$ 代入两个约束方程，可解出 $\lambda_1, \lambda_2$ 关于 $\mu_{\text{target}}$ 的表达式。

### 17.4.2 两基金分离定理

上面的推导揭示了一个深刻的结构:

$$\boxed{\mathbf{w}_{\text{any efficient}} = (1 - \alpha)\mathbf{w}_{\min} + \alpha\mathbf{w}_{\text{tangency}}}$$

任何有效前沿组合都可以表示为**最小方差组合**和**切点组合**的凸组合！这就是**两基金分离定理 (Two-Fund Separation Theorem)**。$\alpha$ 从 $0$ 变到超过 $1$，组合从最保守滑到最激进。用微型示例 $A=50, B=5, C=1.5, r_f=2\%$ 来感受 $\alpha$ 变化时组合的具体变化:

| $\alpha$ | 期望收益 $\mu_p$ | 波动率 $\sigma_p$ | 夏普比率 | 说明 |
|----------|-----------------|------------------|---------|------|
| $0$ | 10.00% | 14.14% | 0.57 | 纯最小方差——完全不关心收益，只求最稳 |
| $0.3$ | 17.50% | 16.01% | 0.97 | 保守型——夏普已接近 1.0 |
| $0.5$ | 22.50% | 18.87% | 1.09 | 均衡配置——波动和收益各半 |
| $0.7$ | 27.50% | 22.50% | 1.13 | 偏激进——夏普接近最大值 |
| $1.0$ | 35.00% | 28.72% | **1.15** | 纯切点——夏普比率达到最大 |
| $1.2$ | 40.00% | 33.17% | 1.15 | 借 20% 资金加杠杆——夏普开始下降 |
| $1.5$ | 47.50% | 40.08% | 1.14 | 借 50% 资金加杠杆——波动骤增 |

![两基金分离: 左图展示alpha从0到1.5在有效前沿上的对应位置, 右图展示波动率和夏普比率随alpha的变化趋势。](images/ch17_fig2_two_fund_separation.png)

**读图要点**: 左图中 $\alpha=0$ (蓝色) 是有效前沿最左端——波动率最低。沿红色曲线向右上移动，每增加 0.1 的 $\alpha$ 组合向切点组合靠近一步。$\alpha=1$ (绿色菱形) 是夏普比率最大处——从该点出发的射线与前沿相切，斜率 $=(\mu_{\text{tang}}-r_f)/\sigma_{\text{tang}}$。$\alpha>1$ (橙色方块) 意味着借钱加杠杆，组合沿前沿继续向右延伸但夏普开始下降——额外的风险没有换来等比例的额外收益。右图直观展示了这一转折: 夏普比率在 $\alpha=1$ 处见顶后缓慢回落。

**极端情况**: $\alpha<0$ 意味着**做空切点组合**借来资金增配最小方差——极度保守到愿意牺牲收益换取更低波动。$\alpha \to \infty$ 意味着无限加杠杆——波动率趋于无穷但夏普继续下降，收益增长赶不上风险增长。

在 $\sigma$-$\mu$ 平面上，把这些组合画出来就得到一条**双曲线**——代入 $\mathbf{w}^* = \mathbf{\Sigma}^{-1}(\lambda_1\mathbf{1} + \lambda_2\boldsymbol{\mu})$ 到 $\sigma^2 = \mathbf{w}^T\mathbf{\Sigma}\mathbf{w}$ 并消去 $\lambda_1, \lambda_2$:

$$\boxed{\sigma^2(\mu) = \frac{A\mu^2 - 2B\mu + C}{AC - B^2}}$$

其中 $A = \mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1} > 0$, $B = \mathbf{1}^T\mathbf{\Sigma}^{-1}\boldsymbol{\mu}$, $C = \boldsymbol{\mu}^T\mathbf{\Sigma}^{-1}\boldsymbol{\mu} > 0$，且 $AC - B^2 > 0$（Cauchy-Schwarz 不等式保证）。这条双曲线的顶点就是最小方差组合 $(\sigma_{\min}, \mu_{\min}) = (1/\sqrt{A}, B/A)$，右支向上延伸代表"承担更多风险换取更高收益"。

> **为什么是双曲线而非抛物线？** 因为 $\sigma$ 和 $\mu$ 都是 $\mathbf{w}$ 的二次型，且 $\mathbf{w}$ 在两个线性约束下只有一维自由度。参数方程消去后，$\sigma^2$ 是 $\mu$ 的二次函数——这正是双曲线的代数特征。

### 17.4.3 量化实战: 用真实A股数据画出有效前沿

代码分五步将 17.4.1-17.4.2 的解析公式落地为可视化:

1. **计算三个关键标量**: $A = \mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1}$、$B = \mathbf{1}^T\mathbf{\Sigma}^{-1}\boldsymbol{\mu}$、$C = \boldsymbol{\mu}^T\mathbf{\Sigma}^{-1}\boldsymbol{\mu}$。这三个数完整刻画了有效前沿的形状——$A$ 控制最小方差位置，$B$ 和 $C$ 联合控制前沿的弯曲程度。
2. **构造两个"基金"**: 最小方差组合 $\mathbf{w}_{\min} = \mathbf{\Sigma}^{-1}\mathbf{1}/A$ 和切点组合 $\mathbf{w}_{\text{tangency}} \propto \mathbf{\Sigma}^{-1}(\boldsymbol{\mu} - r_f\mathbf{1})$。两基金分离定理保证任何有效组合都是这两个的线性组合。
3. **解析生成前沿曲线**: 对一系列目标收益 $\mu_{\text{target}}$，用公式 $\sigma^2 = (A\mu^2 - 2B\mu + C)/(AC-B^2)$ 计算对应的最小方差。这条双曲线就是有效前沿。
4. **蒙特卡洛对比**: 生成2000组随机权重（归一化到和为1），计算每组的 $(\sigma, \mu)$，用灰色散点展示"所有可达组合"的区域。有效前沿应该是这个区域的**左上边界**。
5. **标注个股**: 每只股票自身的 $(\sigma_i, \mu_i)$ 用方块标注——它们通常远在有效前沿之下，直观验证"分散化提升风险-收益比"。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

csv_path = 'data/stock_data_50_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
selected = ['000002.SZ', '600519.SH', '300750.SZ', '000858.SZ',
            '601398.SH', '002415.SZ', '000725.SZ', '300059.SZ',
            '002230.SZ', '600030.SH']

df_recent = df[df['time'] >= '2024-06-01'].copy()
pivot = df_recent.pivot(index='time', columns='thscode', values='close')[selected]
rets = np.log(pivot / pivot.shift(1)).dropna()

mu = rets.mean().values * 252     # 年化期望收益
Sigma = rets.cov().values * 252    # 年化协方差
N = len(selected)
ones = np.ones(N)

# 三个关键标量
Sigma_inv = np.linalg.inv(Sigma)
A = ones @ Sigma_inv @ ones
B = ones @ Sigma_inv @ mu
C = mu @ Sigma_inv @ mu

# 最小方差组合
w_minvar = Sigma_inv @ ones / A
mu_minvar = w_minvar @ mu
sigma_minvar = np.sqrt(1.0 / A)

# 切点组合 (最大夏普比率, 假设无风险利率=0.02)
rf = 0.02
w_tangency = Sigma_inv @ (mu - rf * ones)
w_tangency = w_tangency / np.sum(w_tangency)
mu_tangency = w_tangency @ mu
sigma_tangency = np.sqrt(w_tangency @ Sigma @ w_tangency)

# 有效前沿: 给定的 mu_target, 计算最小方差
mu_targets = np.linspace(mu_minvar, mu.max() * 1.1, 100)
sigma_frontier = np.zeros(len(mu_targets))
for i, mut in enumerate(mu_targets):
    # 解 lambda1, lambda2 然后求 w*
    # 利用公式: sigma^2 = (A*mu^2 - 2*B*mu + C)/(A*C - B^2)
    sigma_frontier[i] = np.sqrt((A*mut**2 - 2*B*mut + C) / (A*C - B**2))

# 随机组合 (蒙特卡洛)
np.random.seed(42)
n_random = 2000
random_mu = np.zeros(n_random)
random_sigma = np.zeros(n_random)
for i in range(n_random):
    w = np.random.randn(N)
    w = w / np.sum(w)
    random_mu[i] = w @ mu
    random_sigma[i] = np.sqrt(w @ Sigma @ w)

# 可视化
fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(random_sigma, random_mu, alpha=0.25, s=3, color='gray', label='随机组合 (2000组)')
ax.plot(sigma_frontier, mu_targets, '-', color='#E74C3C', linewidth=2.5, label='有效前沿 (解析解)')
ax.scatter(sigma_minvar, mu_minvar, color='#2980B9', s=150, marker='o', zorder=10,
           label=f'最小方差组合 (sigma={sigma_minvar:.1%})', edgecolors='black')
ax.scatter(sigma_tangency, mu_tangency, color='#27AE60', s=150, marker='D', zorder=10,
           label=f'切点组合 (夏普={(mu_tangency-rf)/sigma_tangency:.2f})', edgecolors='black')
# 个股
for i in range(N):
    ax.scatter(np.sqrt(Sigma[i,i]), mu[i], color='#E67E22', s=40, marker='s', alpha=0.6)
ax.scatter([], [], color='#E67E22', s=40, marker='s', label='个股')

ax.set_xlabel('年化波动率', fontsize=12); ax.set_ylabel('年化期望收益', fontsize=12)
ax.set_title('Markowitz 有效前沿: 10只A股 (2024.06-2026.05)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"两基金分离定理参数:")
print(f"  A = 1^T Σ^{-1} 1 = {A:.4f}")
print(f"  B = 1^T Σ^{-1} mu = {B:.4f}")
print(f"  C = mu^T Σ^{-1} mu = {C:.6f}")
print(f"  最小方差组合: sigma={sigma_minvar:.2%}, mu={mu_minvar:.2%}")
print(f"  切点组合: sigma={sigma_tangency:.2%}, mu={mu_tangency:.2%}, "
      f"Sharpe={(mu_tangency-rf)/sigma_tangency:.2f}")

**运行结果**:
```
两基金分离定理参数:
  A = 1^T Sigma^{-1} 1 = 57.0291
  B = 1^T Sigma^{-1} mu = 7.4831
  C = mu^T Sigma^{-1} mu = 8.0781
  最小方差组合: sigma=13.24%, mu=13.12%
  切点组合: sigma=44.04%, mu=125.01%, Sharpe=2.79
```

> **注意**: 切点组合的年化收益高达125%、夏普2.79——这显然是样本内过拟合的结果。$\boldsymbol{\mu}$ 的估计误差远大于 $\mathbf{\Sigma}$ 的估计误差，将样本均值直接代入 Markowitz 会产生极端权重。实践中用**收缩估计**、**Black-Litterman 模型**或**只优化风险不优化收益**(最小方差)来避免此问题。这也正是第14章 Ledoit-Wolf 收缩和第18章鲁棒优化的动机。

---

## 17.5 不等式约束与 KKT 条件简介

拉格朗日乘子法处理等式约束。如果约束是不等式——如 $w_i \geq 0$（禁止做空）——需要推广到 **KKT 条件 (Karush-Kuhn-Tucker Conditions)**:

对于问题 $\min f(\mathbf{x})$ s.t. $g_i(\mathbf{x}) \leq 0$:

1. **梯度条件**: $\nabla f + \sum \mu_i \nabla g_i = \mathbf{0}$
2. **互补松弛**: $\mu_i \cdot g_i(\mathbf{x}^*) = 0$——如果约束不起作用 ($g_i < 0$)，$\mu_i$ 必须为零；如果约束起作用 ($g_i = 0$)，$\mu_i \geq 0$
3. **原始可行性**: $g_i(\mathbf{x}^*) \leq 0$
4. **对偶可行性**: $\mu_i \geq 0$

**互补松弛的金融直觉**: 如果禁止做空约束 $w_i \geq 0$ 在最优解处是"紧"的 ($w_i^* = 0$)，则对应的乘子 $\mu_i > 0$——这说明"如果能做空这只股票，目标函数还能进一步优化"。如果 $w_i^* > 0$（约束不起作用），则 $\mu_i = 0$。

第18章将用 `cvxpy` 在不等式约束下真正求解 Markowitz 优化，届时你会看到 KKT 条件在代码中的自动满足。本章的等式约束推导是理解那一步的基础。

---

## 17.6 核心公式速查

> 本节是前述各节公式的集中汇总, 供复习和查阅使用.

| 概念 | 公式 | 量化意义 |
|------|------|---------|
| 拉格朗日函数 | $\mathcal{L}(\mathbf{x},\lambda) = f(\mathbf{x}) - \lambda g(\mathbf{x})$ | 将约束吸收进目标函数的统一方法 |
| 梯度平行条件 | $\nabla f = \lambda \nabla g$ | 最优解处目标与约束的梯度共线 |
| 拉格朗日乘子 | $\lambda^* = \partial f^*/\partial c$ | 约束放宽一单位的边际收益——影子价格 |
| 最小方差组合 | $\mathbf{w}_{\min} = \mathbf{\Sigma}^{-1}\mathbf{1}/(\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$ | 给定协方差矩阵的理论最优低风险组合 |
| 最小方差 | $\sigma_{\min}^2 = 1/(\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$ | 所有全额投资组合中可达的最低波动率 |
| 有效前沿权重 | $\mathbf{w}^* = \mathbf{\Sigma}^{-1}(\lambda_1\mathbf{1} + \lambda_2\boldsymbol{\mu})$ | 任何有效组合都是两个"基金"的线性组合 |
| 有效前沿曲线 | $\sigma^2(\mu) = \frac{A\mu^2-2B\mu+C}{AC-B^2}$ | 在 sigma-mu 平面上是一条双曲线 |
| 两基金分离 | $\mathbf{w} = (1-\alpha)\mathbf{w}_{\min} + \alpha\mathbf{w}_{\text{tangency}}$ | 投资者只需持两个基金 + 调整配比 |
| KKT 互补松弛 | $\mu_i \cdot g_i(\mathbf{x}^*) = 0$ | 约束不起作用时乘子为零 |

---

## 17.7 本章小结

| 概念 | 核心公式 | 量化意义 |
|------|---------|---------|
| 拉格朗日乘子法 | $\mathcal{L} = f - \lambda g$, 令 $\nabla\mathcal{L}=0$ | 等式约束优化的标准求解框架 |
| 梯度平行 | $\nabla f = \lambda \nabla g$ | 最优解处约束面与目标函数相切 |
| 最小方差推导 | $\mathbf{\Sigma}\mathbf{w} = \lambda\mathbf{1}$ → $\mathbf{w}_{\min}$ | 三步导出组合优化的第一个经典解 |
| 有效前沿 | $\mathbf{w}^* = \mathbf{\Sigma}^{-1}(\lambda_1\mathbf{1}+\lambda_2\boldsymbol{\mu})$ | 所有最优组合的统一参数化 |
| 两基金分离 | 最小方差 + 切点 = 所有有效组合 | 复杂优化简化为两资产配比问题 |
| KKT 条件 | 互补松弛: $\mu_i g_i = 0$ | 不等式约束下的极值判定 |

**从本章走向下一章**:
- 第18章将用 `cvxpy` 求解带不等式约束（禁止做空、单资产上限、行业敞口控制）的 Markowitz 优化。本章的解析解依赖 $\mathbf{\Sigma}^{-1}$ 存在且没有不等式约束——第18章将用数值方法突破这些限制，你将看到凸优化算法在真实量化工作流中的完整应用。

---

## 17.8 练习题

### 数学推导

**题1——拉格朗日乘子法的几何证明**: 在约束 $g(x,y)=x^2+y^2-1=0$ 下最大化 $f(x,y)=x+y$。

(a) 用拉格朗日乘子法解出最优解 $(x^*, y^*)$ 和乘子 $\lambda^*$。

(b) 验证在最优解处 $\nabla f$ 与 $\nabla g$ 确实平行。计算两者的比值是否等于 $\lambda^*$。

(c) 如果约束改为 $x^2+y^2=4$，最优解和乘子如何变化？$\lambda^*$ 是变大了还是变小了？解释经济含义。

**题2——最小方差组合的拉格朗日推导**: 

(a) 对 $\mathcal{L} = \frac{1}{2}\mathbf{w}^T\mathbf{\Sigma}\mathbf{w} - \lambda(\mathbf{w}^T\mathbf{1} - 1)$，验证 $\nabla_{\mathbf{w}}\mathcal{L} = \mathbf{\Sigma}\mathbf{w} - \lambda\mathbf{1}$。

(b) 证明 $\mathbf{w}_{\min}$ 的组合方差为 $\sigma_{\min}^2 = 1/(\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$。

(c) 如果额外加入无风险资产（收益 $r_f$，权重 $w_0$，满足 $w_0 + \mathbf{w}^T\mathbf{1} = 1$），推导新的最优解表达式。

**题3——有效前沿的参数方程**:

(a) 从 $\mathbf{w}^* = \mathbf{\Sigma}^{-1}(\lambda_1\mathbf{1} + \lambda_2\boldsymbol{\mu})$ 出发，代入两个约束方程，推导出 $\lambda_1, \lambda_2$ 关于 $\mu_{\text{target}}$ 的线性表达式。

(b) 利用 $\sigma^2 = \mathbf{w}^T\mathbf{\Sigma}\mathbf{w}$，证明 $\sigma^2$ 是 $\mu_{\text{target}}$ 的二次函数——即有效前沿是双曲线。

### 编程实践

**题4——最小方差组合的样本外表现**: 基于17.3.3的代码框架，将数据分为训练期(2024-06-01 ~ 2025-05-31)和测试期(2025-06-01 ~ 2026-05-29)。

(a) 在训练期估计 $\mathbf{\Sigma}$ 并计算 $\mathbf{w}_{\min}$，在测试期计算该组合和等权组合的净值曲线。对比两者的年化波动率和夏普比率。

(b) 最小方差组合在样本外的波动率降低幅度与样本内是否一致？讨论 $\mathbf{\Sigma}$ 估计误差对组合绩效的影响。试用第14章的 Ledoit-Wolf 收缩估计替代样本协方差，观察改进效果。

**题5——有效前沿的滚动构建**: 

(a) 每次用过去252个交易日的数据估计 $\boldsymbol{\mu}$ 和 $\mathbf{\Sigma}$，构建有效前沿，然后持有最小方差组合21个交易日（约1个月）。滚动前进，记录整个样本外期间(2023-06至2026-05)的绩效。

(b) 对比滚动最小方差组合与滚动等权组合的累计收益和回撤。最小方差策略是否在市场下跌时提供了更好的保护？

---

## 17.9 参考文献

1. **Markowitz, H.** (1952). Portfolio selection. *The Journal of Finance*, 7(1), 77-91. （均值-方差框架的原始论文——两页纸改变了整个金融行业。本章17.3和17.4的全部公式都可以追溯到这篇论文）

2. **Merton, R. C.** (1972). An analytic derivation of the efficient portfolio frontier. *Journal of Financial and Quantitative Analysis*, 7(4), 1851-1872. （有效前沿解析推导的经典文献——$A, B, C$ 参数化形式来自此文）

3. **Boyd, S., & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. （第5章系统讲解拉格朗日对偶理论和 KKT 条件——本章17.5内容的理论基础）

4. **Grinold, R. C., & Kahn, R. N.** (1999). *Active Portfolio Management* (2nd ed.). McGraw-Hill. （第4章将拉格朗日乘子法置于量化组合优化的全流程框架中——从 $\mathbf{\Sigma}^{-1}\mathbf{1}$ 到风险预算的实践）

5. **Meucci, A.** (2005). *Risk and Asset Allocation*. Springer. （第6章将有效前沿推广到非正态收益、鲁棒优化和贝叶斯框架——本章古典 Markowitz 的自然延伸）

---

> **愿我们都能在数字与代码之间, 找到理解市场的那把钥匙.**
>
> *数学的理解没有捷径, 量化的能力无法外包.*